# 🚀 AnyProjector — Phase 2: Q-Former 128q + LoRA + Unfreeze Encoder

Train AnyProjector Q-Former to align Whisper audio embeddings with LLM text space.

**Variant:** `qformer_full`
- LoRA: ✅
- Unfreeze Encoder: Last 4 layers

**Pipeline:** Audio → Whisper Encoder → Q-Former Projector → LLM → Loss


In [ ]:
# Install dependencies
!pip install -q transformers datasets torch accelerate peft


In [ ]:
# Verify GPU
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('⚠️ No GPU! Go to Runtime → Change runtime type → GPU')
print(f'PyTorch: {torch.__version__}')


## ⚙️ Configuration
Edit these settings before running.

In [ ]:
# ============================================
# 🔧 VARIANT: Q-Former 128q + LoRA + Unfreeze Encoder
# ============================================
ENCODER_ID    = "openai/whisper-small"
LLM_ID        = "Qwen/Qwen2.5-1.5B-Instruct"

# Datasets: list of (hf_name, transcript_field)
DATASETS = [
    ("doof-ferb/Speech-MASSIVE_vie", "utt"),
    ("doof-ferb/fpt_fosd", "transcription"),
]
MAX_SAMPLES = 15000  # Per dataset limit (0 = all)

NUM_EPOCHS  = 30
BATCH_SIZE  = 16    # A100: 16-32, T4: 4
LR          = 1e-4
GRAD_ACCUM  = 2     # Effective batch = BATCH_SIZE * GRAD_ACCUM
SAVE_DIR    = "checkpoints/phase2/qformer_full"
PATIENCE    = 7     # Early stopping patience
MIN_DELTA   = 0.01  # Min improvement to count
PROMPT      = "Phiên âm đoạn audio sau bằng tiếng Việt:"
RESUME_FROM = None  # Set to checkpoint path to resume
NUM_WORKERS = 4     # Parallel data loading
PRELOAD_RAM = True  # Cache all audio in RAM

# --- LoRA ---
LORA_ENABLED = True
LORA_RANK    = 8
LORA_ALPHA   = 16

# --- Encoder ---
UNFREEZE_ENCODER_LAYERS = 4  # 0 = all frozen


## 📁 Mount Google Drive
Mount Drive trước để auto-backup checkpoint sau khi train xong.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BACKUP_DIR = '/content/drive/MyDrive/AnyProjector/checkpoints/phase2/qformer_full'
os.makedirs(BACKUP_DIR, exist_ok=True)
print(f'✅ Backup folder ready: {BACKUP_DIR}')


## 🧱 AnyProjector Q-Former Module

In [ ]:
"""
projector.py - Kiến trúc mạng AnyProjector (Q-Former).

Q-Former (BLIP-2 style) bridge giữa Audio Encoder và LLM.
Dùng learnable query tokens + cross-attention để nén encoder output
thành số lượng tokens cố định, bất kể audio length.

Kiến trúc:
    ┌──────────────────────────────────┐
    │  Encoder Output (1500, enc_dim)  │  ← Whisper (pad 30s)
    │  + Attention Mask                │  ← Chỉ real tokens, bỏ pad
    └──────────────┬───────────────────┘
                   │  Key, Value
    ┌──────────────▼───────────────────┐
    │  Learnable Queries (64, qf_dim)  │
    │  ┌─────────────────────────────┐ │
    │  │ Self-Attention              │ │
    │  │ Cross-Attention (to encoder)│ │
    │  │ Feed-Forward Network        │ │
    │  └─────────────────────────────┘ │
    │         × num_layers             │
    └──────────────┬───────────────────┘
                   │
    ┌──────────────▼───────────────────┐
    │  Output Projection               │
    │  Linear(qf_dim → llm_dim)        │
    └──────────────┬───────────────────┘
                   │
                   ▼
    (batch, 64, llm_dim) → LLM inputs_embeds
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class QFormerLayer(nn.Module):
    """Single Q-Former layer: Self-Attn → Cross-Attn → FFN."""

    def __init__(self, qformer_dim: int, encoder_dim: int, num_heads: int = 8,
                 ffn_ratio: int = 4):
        super().__init__()

        # Self-Attention (queries attend to each other)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=qformer_dim, num_heads=num_heads, batch_first=True,
        )
        self.self_attn_norm = nn.LayerNorm(qformer_dim)

        # Cross-Attention (queries attend to encoder output)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=qformer_dim, num_heads=num_heads,
            kdim=encoder_dim, vdim=encoder_dim, batch_first=True,
        )
        self.cross_attn_norm = nn.LayerNorm(qformer_dim)

        # Feed-Forward Network
        ffn_hidden = qformer_dim * ffn_ratio
        self.ffn = nn.Sequential(
            nn.Linear(qformer_dim, ffn_hidden),
            nn.GELU(),
            nn.Linear(ffn_hidden, qformer_dim),
        )
        self.ffn_norm = nn.LayerNorm(qformer_dim)

    def forward(self, queries: torch.Tensor, encoder_out: torch.Tensor,
                encoder_mask: torch.Tensor | None = None) -> torch.Tensor:
        """
        Args:
            queries: (batch, num_queries, qformer_dim)
            encoder_out: (batch, enc_seq_len, encoder_dim)
            encoder_mask: (batch, enc_seq_len) — True = pad (ignored),
                          False = real token. Used as key_padding_mask.
        Returns:
            queries: (batch, num_queries, qformer_dim)
        """
        # Self-Attention + residual
        q = self.self_attn_norm(queries)
        q, _ = self.self_attn(q, q, q)
        queries = queries + q

        # Cross-Attention + residual
        q = self.cross_attn_norm(queries)
        q, _ = self.cross_attn(
            query=q, key=encoder_out, value=encoder_out,
            key_padding_mask=encoder_mask,
        )
        queries = queries + q

        # FFN + residual
        queries = queries + self.ffn(self.ffn_norm(queries))

        return queries


class AnyProjector(nn.Module):
    """Q-Former Projector — bridge giữa Audio Encoder và LLM.

    Dùng learnable query tokens + cross-attention để nén encoder output
    thành số lượng tokens cố định. Hỗ trợ attention mask để bỏ qua
    padding tokens từ encoder (Whisper pad 30s).

    Args:
        encoder_dim: Hidden size của Audio Encoder (e.g. 768, 1024).
        llm_dim: Hidden size của LLM (e.g. 1536, 3072).
        num_queries: Số learnable query tokens (output length).
        qformer_dim: Hidden dim bên trong Q-Former.
        num_layers: Số Q-Former layers (self-attn + cross-attn + FFN).
        num_heads: Số attention heads.
    """

    def __init__(self, encoder_dim: int, llm_dim: int,
                 num_queries: int = 64, qformer_dim: int = 768,
                 num_layers: int = 2, num_heads: int = 8):
        super().__init__()

        self.encoder_dim = encoder_dim
        self.llm_dim = llm_dim
        self.num_queries = num_queries
        self.qformer_dim = qformer_dim

        # Learnable query tokens
        self.query_tokens = nn.Parameter(
            torch.randn(1, num_queries, qformer_dim) * 0.02
        )

        # Q-Former transformer layers
        self.layers = nn.ModuleList([
            QFormerLayer(qformer_dim, encoder_dim, num_heads)
            for _ in range(num_layers)
        ])

        # Final LayerNorm before projection
        self.output_norm = nn.LayerNorm(qformer_dim)

        # Project Q-Former dim → LLM dim
        self.output_proj = nn.Linear(qformer_dim, llm_dim)

    def forward(self, encoder_output: torch.Tensor,
                encoder_mask: torch.Tensor | None = None) -> torch.Tensor:
        """Forward pass qua Q-Former.

        Args:
            encoder_output: Output từ Audio Encoder.
                Shape: (batch, enc_seq_len, encoder_dim)
            encoder_mask: Padding mask cho encoder tokens.
                Shape: (batch, enc_seq_len)
                True = padding (bỏ qua), False = real token.
                None = không mask (attend tất cả).

        Returns:
            projected_queries: Output embeddings cho LLM.
                Shape: (batch, num_queries, llm_dim)
        """
        batch_size = encoder_output.shape[0]

        # Expand queries for batch
        queries = self.query_tokens.expand(batch_size, -1, -1)

        # Pass through Q-Former layers
        for layer in self.layers:
            queries = layer(queries, encoder_output, encoder_mask)

        # Normalize + project to LLM space
        queries = self.output_norm(queries)
        projected = self.output_proj(queries)

        return projected

    def count_parameters(self) -> int:
        """Count total trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def __repr__(self) -> str:
        params = self.count_parameters()
        n_layers = len(self.layers)
        return (
            f"AnyProjector(Q-Former)\n"
            f"  encoder_dim={self.encoder_dim}, llm_dim={self.llm_dim}\n"
            f"  queries={self.num_queries}, qformer_dim={self.qformer_dim}\n"
            f"  layers={n_layers}, output=Linear({self.qformer_dim}->{self.llm_dim})\n"
            f"  trainable_params={params:,}"
        )


## 📦 Imports

In [ ]:
import gc
import json
import logging
import math
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset as hf_load_dataset
from torch.utils.data import Dataset, DataLoader, random_split

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logger = logging.getLogger("phase2")
logger.setLevel(logging.INFO)


## 🔧 Config Dataclass

In [ ]:
@dataclass
class Phase2Config:
    encoder_id: str = ENCODER_ID
    llm_id: str = LLM_ID
    datasets: tuple = tuple(DATASETS)
    max_samples_per_dataset: int = MAX_SAMPLES
    val_split: float = 0.1
    sample_rate: int = 16000
    max_audio_seconds: float = 30.0
    num_queries: int = 128
    qformer_dim: int = 768
    qformer_layers: int = 2
    qformer_heads: int = 8
    lora_enabled: bool = LORA_ENABLED
    lora_rank: int = LORA_RANK
    lora_alpha: int = LORA_ALPHA
    lora_target_modules: tuple = ("q_proj", "v_proj")
    unfreeze_encoder_layers: int = UNFREEZE_ENCODER_LAYERS
    num_epochs: int = NUM_EPOCHS
    batch_size: int = BATCH_SIZE
    learning_rate: float = LR
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    max_grad_norm: float = 1.0
    gradient_accumulation_steps: int = GRAD_ACCUM
    prompt_text: str = PROMPT
    save_dir: str = SAVE_DIR
    save_every: int = 5
    early_stopping_patience: int = PATIENCE
    early_stopping_min_delta: float = MIN_DELTA
    resume_from: str = RESUME_FROM
    num_workers: int = NUM_WORKERS
    preload_ram: bool = PRELOAD_RAM


## 📊 Dataset

In [ ]:
class Phase2Dataset(Dataset):
    """Dataset cho Phase 2 Alignment.

    Nhận list các entries đã chuẩn hóa: [{"audio": {...}, "transcript": str}, ...]
    Hỗ trợ preload toàn bộ audio vào RAM để loại bỏ decode overhead.
    """

    def __init__(self, entries: list, sample_rate: int = 16000,
                 max_audio_seconds: float = 30.0, preload_ram: bool = False):
        self.entries = entries
        self.sample_rate = sample_rate
        self.max_samples = int(max_audio_seconds * sample_rate)

        logger.info(f"Dataset: {len(entries)} samples")

        # Preload all audio into RAM as tensors
        self.cache = None
        if preload_ram:
            logger.info("Preloading audio to RAM...")
            self.cache = []
            for i, entry in enumerate(entries):
                waveform = self._process_audio(entry["audio"])
                self.cache.append(waveform)
                if (i + 1) % 1000 == 0:
                    logger.info(f"  preloaded {i+1}/{len(entries)}")
            ram_mb = sum(w.nbytes for w in self.cache) / 1024**2
            logger.info(f"  Done! {len(self.cache)} samples cached ({ram_mb:.0f} MB RAM)")

    def __len__(self):
        return len(self.entries)

    def _process_audio(self, audio_feature: dict) -> torch.Tensor:
        """Convert HF audio feature to processed waveform tensor."""
        waveform = torch.from_numpy(audio_feature["array"].astype(np.float32))
        sr = audio_feature["sampling_rate"]

        # Mono
        if waveform.dim() > 1:
            waveform = waveform.mean(dim=-1)

        # Resample if needed
        if sr != self.sample_rate:
            import torchaudio
            resampler = torchaudio.transforms.Resample(sr, self.sample_rate)
            waveform = resampler(waveform.unsqueeze(0)).squeeze(0)

        # Truncate
        if waveform.shape[0] > self.max_samples:
            waveform = waveform[:self.max_samples]

        return waveform

    def __getitem__(self, idx):
        if self.cache is not None:
            waveform = self.cache[idx]
        else:
            waveform = self._process_audio(self.entries[idx]["audio"])

        return {
            "waveform": waveform,
            "transcript": self.entries[idx]["transcript"],
        }


def collate_fn(batch):
    """Pad waveforms to same length in batch."""
    waveforms = [s["waveform"] for s in batch]
    transcripts = [s["transcript"] for s in batch]
    lengths = torch.tensor([w.shape[0] for w in waveforms])

    waveforms_padded = nn.utils.rnn.pad_sequence(
        waveforms, batch_first=True, padding_value=0.0
    )
    return {
        "waveforms": waveforms_padded,
        "lengths": lengths,
        "transcripts": transcripts,
    }


## 🏋️ Trainer

In [ ]:
class Phase2Trainer:
    """Trainer cho Phase 2 Alignment.

    Quản lý toàn bộ lifecycle: load models, train loop, checkpoint.
    """

    def __init__(self, config: Phase2Config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        logger.info(f"Device: {self.device}")

        # Will be set during setup
        self.encoder = None
        self.projector = None
        self.llm = None
        self.tokenizer = None
        self.processor = None
        self.embed_layer = None
        self.llm_dtype = None
        self.optimizer = None
        self.scheduler = None
        self.global_step = 0
        self.start_epoch = 0

    def setup_models(self):
        """Load tất cả models, apply LoRA / unfreeze as configured."""
        from transformers import (
            WhisperModel, WhisperProcessor,
            AutoModelForCausalLM, AutoTokenizer, AutoConfig,
        )

        # --- 1. Load Whisper Encoder ---
        logger.info(f"Loading encoder: {self.config.encoder_id}")
        self.processor = WhisperProcessor.from_pretrained(self.config.encoder_id)
        whisper_full = WhisperModel.from_pretrained(self.config.encoder_id)
        self.encoder = whisper_full.encoder.eval()
        del whisper_full
        gc.collect()

        # Freeze all encoder params first
        for p in self.encoder.parameters():
            p.requires_grad = False

        # Optionally unfreeze last N encoder layers
        n_unfreeze = self.config.unfreeze_encoder_layers
        if n_unfreeze > 0:
            encoder_layers = self.encoder.layers
            total_layers = len(encoder_layers)
            for layer in encoder_layers[-n_unfreeze:]:
                for p in layer.parameters():
                    p.requires_grad = True
                layer.train()
            unfrozen_params = sum(p.numel() for p in self.encoder.parameters() if p.requires_grad)
            logger.info(f"  🔓 Unfroze last {n_unfreeze}/{total_layers} encoder layers ({unfrozen_params:,} params)")

        encoder_dim = self.encoder.config.d_model
        logger.info(f"  encoder_dim={encoder_dim}, params={sum(p.numel() for p in self.encoder.parameters()):,}")

        # --- 2. Auto-detect LLM dim, create Projector ---
        llm_config = AutoConfig.from_pretrained(self.config.llm_id)
        llm_dim = llm_config.hidden_size if not hasattr(llm_config, 'text_config') else llm_config.text_config.hidden_size
        logger.info(f"LLM dim: {llm_dim} (from {self.config.llm_id})")

        self.projector = AnyProjector(
            encoder_dim=encoder_dim, llm_dim=llm_dim,
            num_queries=self.config.num_queries,
            qformer_dim=self.config.qformer_dim,
            num_layers=self.config.qformer_layers,
            num_heads=self.config.qformer_heads,
        )
        self.projector.to(self.device).train()
        logger.info(f"Projector: {self.projector.count_parameters():,} trainable params")

        # --- 3. Load LLM (bf16, encoder to CPU to free VRAM) ---
        logger.info(f"Loading LLM: {self.config.llm_id}")
        self.encoder.cpu()  # Free VRAM
        torch.cuda.empty_cache()

        self.tokenizer = AutoTokenizer.from_pretrained(self.config.llm_id)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.llm = AutoModelForCausalLM.from_pretrained(
            self.config.llm_id,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        )
        for p in self.llm.parameters():
            p.requires_grad = False
        self.llm.eval()

        # --- 4. Apply LoRA if enabled ---
        if self.config.lora_enabled:
            from peft import LoraConfig, get_peft_model
            lora_config = LoraConfig(
                r=self.config.lora_rank,
                lora_alpha=self.config.lora_alpha,
                target_modules=list(self.config.lora_target_modules),
                bias="none",
                task_type="CAUSAL_LM",
            )
            self.llm = get_peft_model(self.llm, lora_config)
            lora_params = sum(p.numel() for p in self.llm.parameters() if p.requires_grad)
            logger.info(f"  🔗 LoRA applied: rank={self.config.lora_rank}, {lora_params:,} trainable LLM params")

        self.embed_layer = self.llm.get_input_embeddings()
        self.llm_dtype = next(self.llm.parameters()).dtype

        # Move encoder back to GPU
        self.encoder.to(self.device)
        llm_total = sum(p.numel() for p in self.llm.parameters())
        llm_train = sum(p.numel() for p in self.llm.parameters() if p.requires_grad)
        logger.info(f"  LLM: {llm_total:,} total, {llm_train:,} trainable")

        # --- 5. VRAM report ---
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            logger.info(f"  VRAM: {allocated:.1f}GB allocated, {reserved:.1f}GB reserved")

    def setup_optimizer(self, total_steps: int):
        """Create optimizer + warmup cosine scheduler."""
        # Collect all trainable parameters
        self._trainable_params = list(self.projector.parameters())
        trainable_params = self._trainable_params

        # Add LoRA params if enabled
        if self.config.lora_enabled:
            trainable_params += [p for p in self.llm.parameters() if p.requires_grad]

        # Add unfrozen encoder params
        if self.config.unfreeze_encoder_layers > 0:
            trainable_params += [p for p in self.encoder.parameters() if p.requires_grad]

        total_trainable = sum(p.numel() for p in trainable_params)
        logger.info(f"Total trainable params: {total_trainable:,}")

        self.optimizer = torch.optim.AdamW(
            trainable_params,
            lr=self.config.learning_rate,
            weight_decay=self.config.weight_decay,
        )

        warmup_steps = int(total_steps * self.config.warmup_ratio)

        def lr_lambda(step):
            if step < warmup_steps:
                return float(step) / max(1, warmup_steps)
            progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
            return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

        self.scheduler = torch.optim.lr_scheduler.LambdaLR(self.optimizer, lr_lambda)
        logger.info(f"Optimizer: AdamW lr={self.config.learning_rate}, warmup={warmup_steps}/{total_steps}")

    def process_batch(self, batch: dict) -> torch.Tensor:
        """Process 1 batch → return loss (scalar, has grad to projector)."""
        waveforms = batch["waveforms"]  # (B, max_samples)
        transcripts = batch["transcripts"]

        # --- Compute encoder padding mask ---
        # Whisper encoder: 30s audio → 1500 tokens (0.02s/token)
        # Real audio length → real encoder tokens, rest is padding
        encoder_seq_len = 1500  # Whisper fixed output
        samples_per_token = (self.config.max_audio_seconds * self.config.sample_rate) / encoder_seq_len
        encoder_mask = torch.zeros(
            len(waveforms), encoder_seq_len, dtype=torch.bool, device=self.device
        )
        for i, w in enumerate(waveforms):
            real_tokens = min(encoder_seq_len, int(w.shape[0] / samples_per_token))
            encoder_mask[i, real_tokens:] = True  # True = padding → ignore

        # --- Audio → Encoder (frozen, on GPU) ---
        with torch.no_grad():
            audio_inputs = self.processor(
                [w.numpy() for w in waveforms],
                sampling_rate=self.config.sample_rate,
                return_tensors="pt",
                padding="max_length",
            )
            input_features = audio_inputs.input_features.to(self.device)
            encoder_output = self.encoder(input_features).last_hidden_state

        # --- Encoder → Q-Former Projector (trainable) ---
        audio_embeds = self.projector(encoder_output, encoder_mask)  # (B, num_queries, llm_dim)

        # --- Prepare prompt + target embeddings (frozen) ---
        with torch.no_grad():
            prompt_tokens = self.tokenizer(
                self.config.prompt_text,
                return_tensors="pt",
                add_special_tokens=False,
            ).input_ids.to(self.device)
            prompt_embeds = self.embed_layer(prompt_tokens)  # (1, prompt_len, llm_dim)
            # Expand prompt for batch
            prompt_embeds = prompt_embeds.expand(len(transcripts), -1, -1)

            target_tokens = self.tokenizer(
                transcripts,
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
                truncation=True,
                max_length=128,
            ).to(self.device)
            target_embeds = self.embed_layer(target_tokens.input_ids)  # (B, text_len, llm_dim)

        # --- Combine: [prompt | audio | target] → LLM ---
        full_input = torch.cat(
            [prompt_embeds, audio_embeds, target_embeds], dim=1
        ).to(self.llm_dtype)

        # --- Labels: [-100 for prompt+audio, target_ids for text] ---
        batch_size = len(transcripts)
        ignore_len = prompt_embeds.shape[1] + audio_embeds.shape[1]
        ignore_labels = torch.full(
            (batch_size, ignore_len), -100,
            dtype=torch.long, device=self.device,
        )
        labels = torch.cat([ignore_labels, target_tokens.input_ids], dim=1)

        # --- Attention mask ---
        audio_attn = torch.ones(
            (batch_size, ignore_len),
            dtype=torch.long, device=self.device,
        )
        attn_mask = torch.cat([audio_attn, target_tokens.attention_mask], dim=1)

        # --- Forward LLM ---
        outputs = self.llm(
            inputs_embeds=full_input,
            attention_mask=attn_mask,
            labels=labels,
        )

        return outputs.loss

    def _vram_info(self) -> str:
        """Get current VRAM usage string."""
        if torch.cuda.is_available():
            alloc = torch.cuda.memory_allocated() / 1024**3
            total = torch.cuda.get_device_properties(0).total_memory / 1024**3
            return f"{alloc:.1f}/{total:.1f}GB"
        return "N/A"

    def train(self):
        """Full training loop with verbose logging."""
        config = self.config

        # --- Setup ---
        self.setup_models()

        # --- Dataset (multi-source) ---
        entries = []
        for ds_name, transcript_field in config.datasets:
            logger.info(f"Loading: {ds_name} (field='{transcript_field}')")
            ds_splits = []
            for split_name in ["train", "validation", "test"]:
                try:
                    ds = hf_load_dataset(ds_name, split=split_name)
                    ds_splits.append(ds)
                    logger.info(f"  split '{split_name}': {len(ds)} samples")
                except Exception:
                    pass

            if not ds_splits:
                logger.warning(f"  No splits found for {ds_name}, skipping.")
                continue

            from datasets import concatenate_datasets
            merged = concatenate_datasets(ds_splits)

            # Apply max_samples limit
            limit = config.max_samples_per_dataset
            if limit > 0 and len(merged) > limit:
                merged = merged.select(range(limit))
                logger.info(f"  Limited to {limit} samples")

            # Normalize to unified format
            for i in range(len(merged)):
                entries.append({
                    "audio": merged[i]["audio"],
                    "transcript": merged[i][transcript_field],
                })
            logger.info(f"  Added {len(merged)} samples (total: {len(entries)})")

        logger.info(f"Combined dataset: {len(entries)} samples")

        dataset = Phase2Dataset(
            entries,
            sample_rate=config.sample_rate,
            max_audio_seconds=config.max_audio_seconds,
            preload_ram=config.preload_ram,
        )

        # Train/Val split
        val_size = max(1, int(len(dataset) * config.val_split))
        train_size = len(dataset) - val_size
        train_dataset, val_dataset = random_split(
            dataset, [train_size, val_size],
            generator=torch.Generator().manual_seed(42),
        )

        nw = config.num_workers
        train_loader = DataLoader(
            train_dataset, batch_size=config.batch_size,
            shuffle=True, collate_fn=collate_fn,
            num_workers=nw, pin_memory=True,
            persistent_workers=nw > 0,
        )
        val_loader = DataLoader(
            val_dataset, batch_size=config.batch_size,
            shuffle=False, collate_fn=collate_fn,
            num_workers=nw, pin_memory=True,
            persistent_workers=nw > 0,
        )

        logger.info(f"Dataset split: train={train_size}, val={val_size}")
        logger.info(f"Batches/epoch: {len(train_loader)}")

        # --- Optimizer ---
        steps_per_epoch = math.ceil(len(train_loader) / config.gradient_accumulation_steps)
        total_steps = steps_per_epoch * config.num_epochs
        self.setup_optimizer(total_steps)

        # --- Resume ---
        if config.resume_from:
            self.load_checkpoint(config.resume_from)

        # --- Training Loop ---
        logger.info("")
        logger.info("╔" + "═" * 58 + "╗")
        logger.info("║         🚀 PHASE 2 ALIGNMENT TRAINING                    ║")
        logger.info("╠" + "═" * 58 + "╣")
        logger.info(f"║  Encoder:    {config.encoder_id:<43}║")
        logger.info(f"║  LLM:        {config.llm_id:<43}║")
        qf_info = f"{self.projector.count_parameters():>10,} params (Q-Former {config.num_queries}q)"
        logger.info(f"║  Projector:  {qf_info:<43}║")
        logger.info(f"║  Epochs:     {config.num_epochs:<43}║")
        logger.info(f"║  Batch:      {config.batch_size} × {config.gradient_accumulation_steps} accum = {config.batch_size * config.gradient_accumulation_steps} effective{' ' * 24}║")
        logger.info(f"║  LR:         {config.learning_rate:<43}║")
        logger.info(f"║  Steps:      {steps_per_epoch}/epoch, {total_steps} total{' ' * 26}║")
        logger.info(f"║  VRAM:       {self._vram_info():<43}║")
        logger.info("╚" + "═" * 58 + "╝")
        logger.info("")

        best_val_loss = float("inf")
        best_epoch = 0
        patience_counter = 0
        history = []  # Track loss per epoch

        for epoch in range(self.start_epoch, config.num_epochs):
            epoch_num = epoch + 1

            # ============ TRAIN ============
            logger.info(f"━━━ Epoch {epoch_num}/{config.num_epochs} ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
            self.projector.train()
            train_loss_sum = 0.0
            train_steps = 0
            batch_losses = []
            self.optimizer.zero_grad()
            epoch_start = time.time()

            for batch_idx, batch in enumerate(train_loader):
                loss = self.process_batch(batch)

                # Cast loss to fp32 for stable backward
                loss_val = loss.float()
                scaled_loss = loss_val / config.gradient_accumulation_steps
                scaled_loss.backward()

                batch_loss = loss_val.item()
                train_loss_sum += batch_loss
                train_steps += 1
                batch_losses.append(batch_loss)

                # Progress bar (single line, overwrite)
                total = len(train_loader)
                done = batch_idx + 1
                pct = done / total
                bar_len = 25
                filled = int(bar_len * pct)
                bar = "█" * filled + "░" * (bar_len - filled)
                avg_loss = train_loss_sum / train_steps
                lr = self.optimizer.param_groups[0]["lr"]
                elapsed_s = time.time() - epoch_start
                print(
                    f"\r  {bar} {done}/{total} | "
                    f"loss={avg_loss:.4f} | lr={lr:.2e} | "
                    f"{elapsed_s:.0f}s | VRAM {self._vram_info()}",
                    end="", flush=True,
                )

                # Optimizer step
                if (batch_idx + 1) % config.gradient_accumulation_steps == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self._trainable_params, config.max_grad_norm
                    )
                    self.optimizer.step()
                    self.scheduler.step()
                    self.optimizer.zero_grad()
                    self.global_step += 1

            # Flush remaining grads
            if train_steps % config.gradient_accumulation_steps != 0:
                torch.nn.utils.clip_grad_norm_(
                    self._trainable_params, config.max_grad_norm
                )
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()
                self.global_step += 1

            train_avg = train_loss_sum / max(train_steps, 1)
            elapsed = time.time() - epoch_start
            print()  # Newline after progress bar

            # ============ VALIDATION ============
            logger.info(f"  📊 Validating...")
            self.projector.eval()
            val_loss_sum = 0.0
            val_steps = 0

            with torch.no_grad():
                for val_idx, batch in enumerate(val_loader):
                    loss = self.process_batch(batch)
                    val_loss = loss.float().item()
                    val_loss_sum += val_loss
                    val_steps += 1
                    logger.info(f"     val batch {val_idx+1}/{len(val_loader)} | loss={val_loss:.4f}")

            val_avg = val_loss_sum / max(val_steps, 1)

            # ============ EPOCH SUMMARY ============
            history.append({"epoch": epoch_num, "train": train_avg, "val": val_avg})
            overfit_ratio = val_avg / max(train_avg, 1e-8)

            # Early stopping tracking
            if val_avg < best_val_loss - config.early_stopping_min_delta:
                best_val_loss = val_avg
                best_epoch = epoch_num
                patience_counter = 0
                improved = "★ BEST"
                self.save_checkpoint("best", val_avg)
            else:
                patience_counter += 1
                improved = f"wait {patience_counter}/{config.early_stopping_patience}"

            # Overfit warning
            if overfit_ratio > 2.0:
                fit_status = "⚠️ OVERFIT"
            elif overfit_ratio > 1.5:
                fit_status = "😐 MILD"
            elif overfit_ratio > 1.2:
                fit_status = "👍 OK"
            else:
                fit_status = "✅ GOOD"

            logger.info(f"")
            logger.info(f"  ┌─────────────────────────────────────────────┐")
            logger.info(f"  │ Epoch {epoch_num:>2}/{config.num_epochs} Summary{' ' * 27}│")
            logger.info(f"  ├─────────────────────────────────────────────┤")
            logger.info(f"  │ Train Loss:    {train_avg:>8.4f}                      │")
            logger.info(f"  │ Val Loss:      {val_avg:>8.4f}  {improved:<19}│")
            logger.info(f"  │ Overfit Ratio: {overfit_ratio:>8.2f}x {fit_status:<18}│")
            logger.info(f"  │ Best Val:      {best_val_loss:>8.4f}  (epoch {best_epoch:>2}){' ' * 11}│")
            logger.info(f"  │ LR:            {self.optimizer.param_groups[0]['lr']:>8.2e}                      │")
            logger.info(f"  │ Time:          {elapsed:>8.1f}s                     │")
            logger.info(f"  │ Global Step:   {self.global_step:>8}                      │")
            logger.info(f"  │ VRAM:          {self._vram_info():>12}                  │")
            logger.info(f"  └─────────────────────────────────────────────┘")
            logger.info(f"")

            # Save periodic checkpoint
            if (epoch_num) % config.save_every == 0:
                self.save_checkpoint(epoch_num, val_avg)

            # Early stopping check
            if patience_counter >= config.early_stopping_patience:
                logger.info(f"🛑 Early stopping! Val loss không cải thiện sau {config.early_stopping_patience} epochs.")
                logger.info(f"   Best model: epoch {best_epoch}, val_loss={best_val_loss:.4f}")
                self.save_checkpoint(f"early_stop_e{epoch_num}", val_avg)
                break

        # Save final
        self.save_checkpoint("final", val_avg)

        # ============ FINAL SUMMARY ============
        logger.info("")
        logger.info("╔" + "═" * 58 + "╗")
        logger.info("║         🏁 TRAINING COMPLETE                             ║")
        logger.info("╠" + "═" * 58 + "╣")
        logger.info(f"║  Best Val Loss:  {best_val_loss:<39.4f}║")
        logger.info(f"║  Final Train:    {history[-1]['train']:<39.4f}║")
        logger.info(f"║  Total Steps:    {self.global_step:<39}║")
        logger.info(f"║  Checkpoints:    {config.save_dir:<39}║")
        logger.info("╠" + "═" * 58 + "╣")
        logger.info("║  Loss History (last 10 epochs):                          ║")
        for h in history[-10:]:
            bar_len = int(max(0, min(30, (h['train'] / max(history[0]['train'], 1)) * 30)))
            bar = "█" * bar_len + "░" * (30 - bar_len)
            logger.info(f"║  E{h['epoch']:>2} T={h['train']:.3f} V={h['val']:.3f} {bar} ║")
        logger.info("╚" + "═" * 58 + "╝")

        # ============ AUTO-BACKUP TO DRIVE ============
        self._backup_to_drive()

    def _backup_to_drive(self):
        """Auto-backup checkpoints to Google Drive (must be mounted beforehand)."""
        import shutil

        src_dir = Path(self.config.save_dir)
        dst_dir = Path("/content/drive/MyDrive/AnyProjector") / self.config.save_dir

        if not src_dir.exists():
            logger.warning("No checkpoints to backup.")
            return

        # Check Drive mounted
        if not Path("/content/drive").exists():
            logger.info("Google Drive not mounted — skipping backup.")
            logger.info("  (Chạy local hoặc chưa mount Drive trong notebook)")
            return

        # Ensure backup folder exists
        dst_dir.mkdir(parents=True, exist_ok=True)

        # Copy all checkpoints
        count = 0
        for ckpt_file in src_dir.glob("*.pt"):
            shutil.copy2(ckpt_file, dst_dir / ckpt_file.name)
            logger.info(f"  📁 Backed up: {ckpt_file.name} → Drive")
            count += 1

        logger.info(f"✅ {count} checkpoint(s) saved to: {dst_dir}")

        # Disconnect runtime to free GPU
        try:
            from google.colab import runtime
            logger.info("🔌 Disconnecting Colab runtime...")
            runtime.unassign()
        except Exception:
            pass  # Not on Colab or API not available

    def save_checkpoint(self, tag, val_loss=None):
        """Save projector weights + training state + LoRA/encoder if applicable."""
        save_dir = Path(self.config.save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

        ckpt_data = {
            "projector_state_dict": self.projector.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "scheduler_state_dict": self.scheduler.state_dict() if self.scheduler else None,
            "global_step": self.global_step,
            "epoch": tag if isinstance(tag, int) else -1,
            "val_loss": val_loss,
            "config": {
                "encoder_id": self.config.encoder_id,
                "llm_id": self.config.llm_id,
                "encoder_dim": self.projector.encoder_dim,
                "llm_dim": self.projector.llm_dim,
                "num_queries": self.projector.num_queries,
                "lora_enabled": self.config.lora_enabled,
                "unfreeze_encoder_layers": self.config.unfreeze_encoder_layers,
            },
        }

        # Save LoRA adapter weights
        if self.config.lora_enabled:
            lora_state = {k: v for k, v in self.llm.state_dict().items() if "lora" in k}
            ckpt_data["lora_state_dict"] = lora_state

        # Save unfrozen encoder layer weights
        if self.config.unfreeze_encoder_layers > 0:
            encoder_state = {k: v for k, v in self.encoder.state_dict().items()
                             if any(k.startswith(f"layers.{len(self.encoder.layers) - i - 1}")
                                    for i in range(self.config.unfreeze_encoder_layers))}
            ckpt_data["encoder_state_dict"] = encoder_state

        path = save_dir / f"projector_{tag}.pt"
        torch.save(ckpt_data, path)
        logger.info(f"  Checkpoint saved: {path}")

        # Also save as 'latest'
        latest = save_dir / "latest.pt"
        torch.save(torch.load(path, weights_only=False), latest)

    def load_checkpoint(self, path: str):
        """Resume from checkpoint."""
        logger.info(f"Resuming from: {path}")
        ckpt = torch.load(path, map_location=self.device, weights_only=False)

        self.projector.load_state_dict(ckpt["projector_state_dict"])
        if self.optimizer and "optimizer_state_dict" in ckpt:
            self.optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if self.scheduler and ckpt.get("scheduler_state_dict"):
            self.scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        self.global_step = ckpt.get("global_step", 0)
        self.start_epoch = ckpt.get("epoch", 0)
        if isinstance(self.start_epoch, int) and self.start_epoch > 0:
            logger.info(f"  Resumed at epoch {self.start_epoch}, step {self.global_step}")


## 🚀 Run Training

In [ ]:
config = Phase2Config()
print(config)
print()
trainer = Phase2Trainer(config)
trainer.train()


## 💾 Download Best Checkpoint

In [ ]:
from google.colab import files
files.download(f"{SAVE_DIR}/projector_best.pt")
